# Imersão Alura Santander — Nivelamento IA 2026
## Semana 03 — Python Avançado + IA S03 + LangChain + Provider Switcher

Este notebook acompanha a live da Semana 03.
Cada seção Python demonstra o conceito **diretamente dentro de um agente automatizado** (SupportBot),
não em exemplos isolados.

**Estrutura:**
1. Setup — Provider Switcher (Groq ou OpenRouter, 2 linhas de diferença)
2. Fechamento dos pendentes da S02: Tokens/Temperatura e 4 técnicas de Prompt
3. Python S03: Funções → Lambda → Listas/Tuplas → Comprehensions → zip → try/except
4. IA S03: Evolução de modelos, System Prompt, Limite de contexto
5. LangChain: PromptTemplate | ChatGroq/ChatOpenAI | invoke()


---
## ⚙️ Setup — Instalação e Provider Switcher


### Por que usamos o pacote `openai` para acessar o Groq?

A interface da API OpenAI virou o **padrão da indústria**. Groq, OpenRouter, Azure e dezenas de outros providers
implementam a mesma interface. Isso significa que você **troca de provider mudando 2 linhas** — `base_url` e `api_key`.
O restante do código não toca.

Execute a célula abaixo primeiro. Todas as demos seguintes usam as variáveis `client` e `MODEL` definidas aqui.


In [ ]:
# Instalar dependências (só na primeira vez)
!pip install openai langchain langchain-groq langchain-openai --quiet


In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

# ─── ESCOLHA SEU PROVIDER ───────────────────────────────────────────────
PROVIDER = 'groq'   # troque para 'openrouter' e o restante funciona igual
# ────────────────────────────────────────────────────────────────────────

if PROVIDER == 'groq':
    client = OpenAI(
        base_url='https://api.groq.com/openai/v1',
        api_key=userdata.get('GROQ_API_KEY')
    )
    MODEL = 'llama-3.3-70b-versatile'

elif PROVIDER == 'openrouter':
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=userdata.get('OPENROUTER_API_KEY')
    )
    MODEL = 'meta-llama/llama-3.3-70b-instruct'

print(f'✅ Provider: {PROVIDER}')
print(f'   Modelo  : {MODEL}')


---
## 📌 Pendentes da S02 — Sprint de 10 minutos


### Tokens e Temperatura

**Token ≠ palavra.** Em português, 1 palavra ≈ 1,5 a 2 tokens. Isso impacta custo e o limite de contexto do modelo.

**Temperatura** controla aleatoriedade:
- `0.0` → determinístico (mesma resposta toda vez) → ideal para classificação
- `0.3` → levemente variado → ideal para respostas técnicas
- `0.8` → criativo → ideal para email de retenção
- `1.5+` → caótico → evitar em produção


In [ ]:
# Mesmo prompt, temperaturas diferentes — veja a variação
prompt_temp = 'O cliente quer cancelar. Ofereça um desconto para retê-lo. Responda em 1 frase.'

for temp in [0.0, 0.7, 1.5]:
    r = client.chat.completions.create(
        model=MODEL,
        temperature=temp,
        messages=[{'role': 'user', 'content': prompt_temp}]
    )
    print(f'[temp={temp}] {r.choices[0].message.content}')
    print()


### As 4 técnicas de Engenharia de Prompt

| Técnica | Problema que resolve | Quando usar |
|---------|---------------------|-------------|
| **Few-Shot** | Modelo não sabe o padrão de saída | Classificação, formatação, roteamento |
| **Chain-of-Thought** | Erros em problemas de múltiplos passos | Análise, cálculo, diagnóstico |
| **Chain of Verification** | Alucinações e fatos inventados | Respostas sobre produto, legal, médico |
| **Self-Consistency** | Decisão crítica pode ser enviesada | Aprovar reembolso, escalar para humano |


In [ ]:
CASO = "Cliente comprou há 8 dias, produto com defeito leve, sem nota fiscal."

def chamar(prompt, temp=0.0):
    r = client.chat.completions.create(
        model=MODEL, temperature=temp,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.choices[0].message.content

# 1. FEW-SHOT ─────────────────────────────────────────────────────────────
print('=' * 50)
print('1. FEW-SHOT — ensinar padrão por exemplo')
print('=' * 50)
prompt_fewshot = f"""
Decida REEMBOLSAR ou NEGAR com base nos exemplos:

Exemplos:
"Comprou há 2 dias, produto OK, tem nota fiscal" -> REEMBOLSAR (dentro do prazo)
"Comprou há 40 dias, produto quebrado, sem nota" -> NEGAR (fora do prazo + sem nota)
"Comprou há 5 dias, defeito grave, com nota"     -> REEMBOLSAR (defeito + nota + prazo)

Caso: "{CASO}"
Decisão:"""
print(chamar(prompt_fewshot))


In [ ]:
# 2. CHAIN-OF-THOUGHT ────────────────────────────────────────────────────
print('=' * 50)
print('2. CHAIN-OF-THOUGHT — pensar passo a passo')
print('=' * 50)
prompt_cot = f"""
Analise passo a passo e decida REEMBOLSAR ou NEGAR.

Caso: "{CASO}"

Passo 1: Qual o prazo de troca da empresa?
Passo 2: O caso está dentro do prazo?
Passo 3: A gravidade do defeito justifica exceção?
Passo 4: A falta de nota fiscal é impeditivo?
Decisão final e justificativa:"""
print(chamar(prompt_cot))


In [ ]:
# 3. CHAIN OF VERIFICATION ───────────────────────────────────────────────
print('=' * 50)
print('3. CHAIN OF VERIFICATION — corrigir a si mesmo')
print('=' * 50)
prompt_cove = f"""
Caso: "{CASO}"

PASSO 1 — Gere uma decisão inicial (REEMBOLSAR ou NEGAR) com justificativa.
PASSO 2 — Escreva 2 perguntas que verificam as afirmações do Passo 1.
PASSO 3 — Responda cada pergunta apenas com os fatos do caso.
PASSO 4 — Revise a decisão se alguma verificação contradizer o Passo 1."""
print(chamar(prompt_cove))


In [ ]:
# 4. SELF-CONSISTENCY ────────────────────────────────────────────────────
from collections import Counter
print('=' * 50)
print('4. SELF-CONSISTENCY — votação entre respostas')
print('=' * 50)
prompt_sc = f"Caso: \"{CASO}\" Decida REEMBOLSAR ou NEGAR. Responda só a decisão."

votos = []
for i in range(3):
    decisao = chamar(prompt_sc, temp=0.8)
    votos.append('REEMBOLSAR' if 'REEMBOLSAR' in decisao.upper() else 'NEGAR')
    print(f'  Voto {i+1}: {decisao.strip()[:80]}')

contagem = Counter(votos)
vencedor = contagem.most_common(1)[0][0]
print(f'\nVotação: {dict(contagem)} → Decisão por maioria: {vencedor}')


---
## 🐍 Python S03 — cada conceito dentro de um agente


### Abordagem desta seção

Cada conceito Python não é demonstrado isoladamente — ele aparece **construindo uma peça do SupportBot**.
No final desta seção, você terá um agente funcional com classificação, roteamento, histórico, pré-processamento e resiliência.


---
### 1. Funções — o agente como pipeline de etapas

**Conceito:** funções nomeiam etapas e permitem reutilização. Um agente real é um pipeline:
classificar → responder → escalar. Cada etapa vira uma função independente.

> **Vantagem:** se amanhã você quiser trocar a lógica de classificação por ML, só a função `classificar_intencao()` muda.
> O restante do pipeline continua igual.


In [ ]:
# client e MODEL vêm do Provider Switcher

def classificar_intencao(mensagem: str) -> str:
    """Etapa 1: descobre do que o cliente está falando."""
    prompt = (
        f"Classifique em uma palavra: SUPORTE_TECNICO | FINANCEIRO | RETENCAO | OUTROS\n"
        f"Mensagem: '{mensagem}'\n"
        f"Responda apenas com a categoria, sem explicação."
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.choices[0].message.content.strip()


def gerar_resposta(mensagem: str, intencao: str) -> str:
    """Etapa 2: responde com contexto da intenção identificada."""
    system = (
        f'Você é o SupportBot. A intenção do cliente é {intencao}. '
        'Responda em português, de forma direta e empática.'
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.3,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': mensagem}
        ]
    )
    return r.choices[0].message.content


def processar_ticket(mensagem: str) -> dict:
    """Pipeline completo: classifica e responde."""
    intencao = classificar_intencao(mensagem)
    resposta = gerar_resposta(mensagem, intencao)
    return {'mensagem': mensagem, 'intencao': intencao, 'resposta': resposta}


# Teste ao vivo
resultado = processar_ticket('O app não abre desde ontem')
print(f"Intenção : {resultado['intencao']}")
print(f"Resposta : {resultado['resposta']}")


---
### 2. Lambda — pré-processamento de input antes do LLM

**Conceito:** lambda é uma função anônima de uma linha. Em agentes, aparece no pré-processamento:
normalizar, limpar e truncar texto *antes* de enviar ao modelo.

> **Regra prática:** se cabe em uma linha e não precisa de nome externo, use `lambda`.
> Se precisar de `if`, `for` ou `try`, use `def`.


In [ ]:
# Lambdas de pré-processamento
normalizar     = lambda t: t.strip().lower()
truncar        = lambda t, n=500: t[:n]          # evita tokens desnecessários
limpar_quebras = lambda t: t.replace('\n', ' ').replace('\t', ' ')

def preparar_input(mensagem: str) -> str:
    """Pipeline de limpeza: normaliza → trunca → remove quebras."""
    return limpar_quebras(truncar(normalizar(mensagem)))


# Processar fila de tickets com map()
fila_bruta = [
    '  MEU APP NÃO ABRE   ',
    'Fui COBRADO DUAS VEZES\nneste mês',
    'Quero cancelar minha conta',
]

fila_limpa = list(map(preparar_input, fila_bruta))
print('Antes:', fila_bruta)
print('Depois:', fila_limpa)

# Processar cada ticket limpo com o pipeline do agente
print('\n--- Processando fila ---')
for msg in fila_limpa:
    r = processar_ticket(msg)
    print(f"[{r['intencao']}] {msg}")


---
### 3. Listas vs Tuplas — histórico de conversa e configuração

**Conceito:** lista é mutável (cresce, muda); tupla é imutável (fixo em runtime).

| | Lista | Tupla |
|--|-------|-------|
| **Síntaxe** | `[a, b]` | `(a, b)` |
| **Mutável?** | Sim | Não |
| **Uso em agentes** | Histórico de conversa | Configurações fixas |

> **No SupportBot:** o histórico é uma lista que cresce a cada turno.
> As configurações do agente (providers, intenções válidas) são tuplas — comunicam que não devem mudar.


In [ ]:
# Configuração imutável do agente — tupla
INTENCOES_VALIDAS = ('SUPORTE_TECNICO', 'FINANCEIRO', 'RETENCAO', 'OUTROS')
PROVIDERS_ATIVOS  = ('groq', 'openrouter')

# Tentar modificar gera TypeError (experimente descomendando):
# INTENCOES_VALIDAS[0] = 'NOVO'  # TypeError!

# Histórico mutável — lista que cresce a cada mensagem
historico = []

SYSTEM_PROMPT = (
    'Você é o SupportBot. Responda em português. '
    'Seja direto e empático. Nunca invente informações.'
)

def chat_com_memoria(mensagem_usuario: str) -> str:
    """Agente com memória: o modelo vê toda a conversa anterior."""
    historico.append({'role': 'user', 'content': mensagem_usuario})
    mensagens = [{'role': 'system', 'content': SYSTEM_PROMPT}] + historico

    r = client.chat.completions.create(
        model=MODEL, temperature=0.3, messages=mensagens
    )
    resposta = r.choices[0].message.content
    historico.append({'role': 'assistant', 'content': resposta})
    return resposta


# Simular conversa de 3 turnos
# Na 3ª mensagem, o modelo LEMBRA do que foi dito antes
print('Turno 1:', chat_com_memoria('O app não abre desde ontem.'))
print()
print('Turno 2:', chat_com_memoria('Já tentei reinstalar e não resolveu.'))
print()
print('Turno 3:', chat_com_memoria('Qual o próximo passo?'))
print(f'\nMensagens no histórico: {len(historico)}')


---
### 4. List Comprehension — few-shot dinâmico e filtragem de histórico

**Conceito:** cria uma lista aplicando uma expressão a cada item de outra, com filtro opcional.

```python
# Estrutura geral
[expressão for item in iterável if condição]
```

> **No SupportBot:** os exemplos do few-shot são gerados automaticamente por list comprehension.
> Adicionar um novo exemplo à lista **atualiza o prompt sozinho**, sem editar o texto manualmente.


In [ ]:
# Banco de exemplos few-shot — só adicione linhas aqui
exemplos = [
    ('O app trava ao abrir',        'SUPORTE_TECNICO'),
    ('Fui cobrado duas vezes',      'FINANCEIRO'),
    ('Quero cancelar',              'RETENCAO'),
    ('Tem plano para 50 usuários?', 'VENDAS'),
]

# List comprehension gera o bloco de exemplos — prompt atualiza automaticamente
bloco_fewshot = [
    f'"{msg}" -> {label}'
    for msg, label in exemplos
]
print('Exemplos gerados:')
print('\n'.join(bloco_fewshot))


def classificar_com_fewshot(mensagem: str) -> str:
    """Classificador que usa o few-shot gerado automaticamente."""
    exemplos_str = '\n'.join(bloco_fewshot)
    prompt = (
        f'Classifique em: SUPORTE_TECNICO | FINANCEIRO | RETENCAO | VENDAS | OUTROS\n'
        f'Exemplos:\n{exemplos_str}\n'
        f'Mensagem: "{mensagem}"\n'
        f'Classificação:'
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.choices[0].message.content.strip()


# Classificar uma lista de mensagens em uma linha
mensagens_teste = ['Não consigo fazer login', 'Quero upgrade de plano', 'Cobrança indevida']
classificacoes  = [classificar_com_fewshot(m) for m in mensagens_teste]

print('\nResultados:')
for msg, cls in zip(mensagens_teste, classificacoes):
    print(f'  [{cls}] {msg}')


In [ ]:
# Bônus: filtrar histórico com comprehension
# (Usa o historico criado na seção anterior)
msgs_usuario = [m['content'] for m in historico if m['role'] == 'user']
print('Mensagens do usuário no histórico:')
for m in msgs_usuario:
    print(f'  - {m}')


---
### 5. Dict Comprehension — roteamento por intenção

**Conceito:** cria um dicionário a partir de pares chave-valor com uma expressão.

```python
# Estrutura geral
{chave: valor for item in iterável if condição}
```

> **No SupportBot:** cada intenção recebe um system prompt diferente.
> Dict comprehension monta essa tabela de roteamento dinamicamente a partir de um dicionário de instruções.


In [ ]:
# Base de instruções por intenção — só edite aqui
instrucoes = {
    'SUPORTE_TECNICO': 'Peça logs e prints. Proponha soluções passo a passo.',
    'FINANCEIRO':      'Solicite número do pedido. Nunca processe reembolso sem confirmar.',
    'RETENCAO':        'Seja muito empático. Ofereça pausa de plano antes de falar em cancelar.',
    'VENDAS':          'Apresente os planos disponíveis. Pergunte sobre o tamanho da equipe.',
}

# Dict comprehension: montar system prompts completos
BASE = 'Você é o SupportBot. Responda em português. '
system_prompts = {
    intencao: BASE + instrucao
    for intencao, instrucao in instrucoes.items()
}

print('System prompts por intenção:')
for k, v in system_prompts.items():
    print(f'  [{k}]')
    print(f'    {v}')


def responder_roteado(mensagem: str, intencao: str) -> str:
    """Usa o system prompt específico da intenção detectada."""
    system = system_prompts.get(intencao, BASE + 'Ajude o cliente da melhor forma.')
    r = client.chat.completions.create(
        model=MODEL, temperature=0.3,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': mensagem}
        ]
    )
    return r.choices[0].message.content


# Teste: mensagem de retenção
msg_teste = 'Quero cancelar minha conta'
intencao  = classificar_com_fewshot(msg_teste)
resposta  = responder_roteado(msg_teste, intencao)
print(f'\nIntenção : {intencao}')
print(f'Resposta : {resposta}')


---
### 6. zip() — avaliação em batch do agente

**Conceito:** `zip()` emparelha dois (ou mais) iteráveis em pares, de forma sincronizada.

> **No SupportBot:** usamos zip() para comparar o que o classificador detectou com o que era esperado —
> medindo a acurácia do agente de forma automática.


In [ ]:
# Suite de teste: mensagens e intenções esperadas
mensagens_eval    = [
    'O app trava ao abrir',
    'Fui cobrado indevidamente',
    'Quero cancelar minha conta',
    'Tem plano para equipes grandes?',
]
intencoes_esperadas = ['SUPORTE_TECNICO', 'FINANCEIRO', 'RETENCAO', 'VENDAS']

# Rodar o classificador em todas as mensagens
intencoes_detectadas = [classificar_com_fewshot(m) for m in mensagens_eval]

# zip() emparelha os três iteráveis para avaliar
print('=' * 55)
print('RELATÓRIO DE AVALIAÇÃO DO SUPPORTBOT')
print('=' * 55)
acertos = 0
for msg, esperado, detectado in zip(mensagens_eval, intencoes_esperadas, intencoes_detectadas):
    ok = esperado == detectado
    acertos += int(ok)
    status = '✓' if ok else '✗'
    print(f'{status} Esperado: {esperado:<20} Detectado: {detectado:<20}')
    print(f'  "{msg}"')

print(f'\nAcurácia: {acertos}/{len(mensagens_eval)} ({acertos/len(mensagens_eval)*100:.0f}%)')


---
### 7. try / except / finally / raise — agente resiliente

**Conceito:** tratamento de exceção para que o agente não pare na primeira falha.
Em produção, rate limits, quedas de rede e timeouts são normais.

| Bloco | Quando executa | Uso típico |
|-------|---------------|------------|
| `try` | Sempre (bloco principal) | Código que pode falhar |
| `except TipoErro` | Só se o erro ocorrer | Tratar o erro específico |
| `finally` | Sempre, com ou sem erro | Logs, fechar conexões |
| `raise` | Quando você quer relançar | Erros irrecuperáveis |

> ⚠️ **Nunca use `except:` sem tipo.** Isso esconde todos os erros, incluindo bugs de lógica.
> Use `except Exception` como mínimo em produção.


In [ ]:
import time
from openai import RateLimitError, APIConnectionError

def processar_ticket_robusto(mensagem: str, max_tentativas: int = 3) -> dict:
    """Pipeline completo com resiliência para produção."""
    mensagem_limpa = preparar_input(mensagem)   # lambda da seção 2

    for tentativa in range(1, max_tentativas + 1):
        try:
            intencao = classificar_com_fewshot(mensagem_limpa)   # seção 4
            resposta = responder_roteado(mensagem_limpa, intencao)  # seção 5
            return {
                'mensagem':   mensagem,
                'intencao':   intencao,
                'resposta':   resposta,
                'provider':   PROVIDER,
                'tentativas': tentativa
            }

        except RateLimitError:
            espera = 2 ** tentativa   # backoff exponencial: 2, 4, 8 seg
            print(f'[{PROVIDER}] Rate limit. Aguardando {espera}s (tentativa {tentativa})...')
            time.sleep(espera)

        except APIConnectionError as e:
            raise RuntimeError(f'[{PROVIDER}] Falha de conexão: {e}') from e

        finally:
            print(f'  → Tentativa {tentativa} concluída.')  # executa SEMPRE

    # Esgotou tentativas — fallback para humano
    return {
        'mensagem': mensagem, 'intencao': 'ERRO',
        'resposta': 'Desculpe, estamos com instabilidade. Um agente humano vai te atender.',
        'provider': PROVIDER, 'tentativas': max_tentativas
    }


# Agente completo em ação — processa a fila toda
fila_final = [
    '  MEU APP NÃO ABRE  ',
    'Fui cobrado duas vezes este mês',
    'Quero cancelar minha conta',
]

print('=== PROCESSAMENTO EM PRODUÇÃO ===')
resultados = [processar_ticket_robusto(m) for m in fila_final]

print('\n=== RESULTADOS ===')
for r in resultados:
    print(f"[{r['intencao']}] {r['mensagem'][:45]}")
    print(f"  → {r['resposta'][:100]}...")
    print(f"  Provider: {r['provider']} | Tentativas: {r['tentativas']}")
    print()


---
### 🏆 Checkpoint Python S03

Você acabou de construir um agente de suporte completo usando **7 conceitos Python** em sequência:

| Conceito | Papel no SupportBot |
|----------|--------------------|
| Funções | Pipeline: classificar → responder |
| Lambda | Normalizar input antes do LLM |
| Listas / Tuplas | Histórico de conversa / Configuração imutável |
| List Comprehension | Few-shot dinâmico, filtrar histórico |
| Dict Comprehension | Roteamento intenção → system prompt |
| zip() | Avaliação em batch com acurácia |
| try/except/finally | Resiliência com retry e fallback |


---
## 🤖 IA Semana 03 — Evolução de Modelos e System Prompt


### Linha do tempo dos modelos de linguagem

Entender a evolução importa porque o **modelo que você escolhe determina o que o agente consegue fazer**.

| Ano | Modelo | Salto principal |
|-----|--------|----------------|
| 2020 | GPT-3 | Completar texto com qualidade. Revolucionário, mas inconsistente. |
| 2022 | ChatGPT (GPT-3.5 + RLHF) | Qualquer pessoa conversa com IA pela primeira vez. |
| 2023 | GPT-4 | Multimodal, raciocínio muito melhor. Custo alto. |
| 2023 | Llama 2 (Meta) | Open-source. Qualquer pessoa pode rodar localmente. |
| 2024 | GPT-4o | Velocidade do 3.5 com qualidade do 4. Novo padrão. |
| 2024 | Claude 3.5 Sonnet | Referência em codificação. Janela 200K tokens. |
| 2024 | o1 / o3 | "Pensa antes de responder". Mais preciso, não mais rápido. |
| 2024 | **Llama 3.3 70B** | **Qualidade GPT-4 class, gratuito no Groq. É o que usamos.** |

**Os 3 saltos que importam:**
1. **GPT-3 → ChatGPT:** RLHF alinhou o modelo com intenção humana. De "completar texto" para "conversar".
2. **GPT-4 → open-source:** Llama democratizou. Você não precisa da OpenAI para ter qualidade alta.
3. **GPT-4o → o1:** De "gerar rápido" para "raciocinar devagar". Para tarefas complexas, o modelo pensa antes de responder.


---
### System Prompt vs User Message

Confusão muito comum entre iniciantes. **São papéis estruturalmente diferentes:**

| Role | Quando usar | O que coloca |
|------|-------------|-------------|
| `system` | Uma vez, no início | Personalidade, regras, contexto permanente do agente |
| `user` | A cada mensagem | O que o cliente digitou agora |
| `assistant` | Vem do modelo | O que o agente respondeu (entra no histórico) |

> **Regra prática:** tudo que você repetiria em todo prompt vai no `system`.
> O que muda a cada mensagem vai no `user`.


In [ ]:
# Mesmo user message — veja a diferença com e sem system prompt
mensagem_cliente = 'Quero cancelar minha assinatura'

# Sem system prompt
r_sem = client.chat.completions.create(
    model=MODEL, temperature=0.3,
    messages=[{'role': 'user', 'content': mensagem_cliente}]
)
print('SEM system prompt:')
print(r_sem.choices[0].message.content)
print()

# Com system prompt do SupportBot
r_com = client.chat.completions.create(
    model=MODEL, temperature=0.3,
    messages=[
        {'role': 'system', 'content': (
            'Você é o SupportBot da Empresa X. '
            'Responda em português. '
            'Seja empático. '
            'Ofereça alternativas antes de aceitar o cancelamento. '
            'Nunca processe reembolso sem confirmar o pedido.'
        )},
        {'role': 'user', 'content': mensagem_cliente}
    ]
)
print('COM system prompt do SupportBot:')
print(r_com.choices[0].message.content)


---
### Limite de contexto e histórico longo

Todo modelo tem uma **janela de contexto máxima**. Quando o histórico fica muito longo,
as mensagens antigas são cortadas — o modelo 'esquece'.

- **Llama 3.3 70B no Groq:** janela de 128K tokens (~90K palavras)
- **Estratégia simples:** manter só as N mensagens mais recentes
- **Estratégia avançada (S04+):** RAG — recuperar só o contexto relevante


In [ ]:
MAX_HISTORICO = 10

def construir_contexto(historico: list, system_prompt: str) -> list:
    """Retorna apenas as N mensagens mais recentes + system prompt."""
    recentes = historico[-MAX_HISTORICO:]  # fatia das últimas N
    return [{'role': 'system', 'content': system_prompt}] + recentes

# Demonstração
historico_longo = [
    {'role': 'user',      'content': f'Mensagem {i}'}
    for i in range(25)
]

ctx = construir_contexto(historico_longo, 'Você é o SupportBot.')
print(f'Histórico total: {len(historico_longo)} mensagens')
print(f'Contexto enviado: {len(ctx)} mensagens (1 system + {MAX_HISTORICO} recentes)')


---
## 🔗 LangChain — Primeira Cadeia com Provider Switcher


### Por que LangChain?

O código Python que construímos acima funciona perfeitamente.
LangChain adiciona três coisas específicas:

1. **Separação de estrutura e conteúdo:** `PromptTemplate` separa o esqueleto do prompt das variáveis
2. **Composição com `|`:** o operador pipe encadeia componentes — output de um vira input do próximo
3. **Portabilidade futura:** a mesma chain funciona para RAG, agentes com memória, multi-step — você aprende a linguagem uma vez

```python
# O operador | é o LCEL (LangChain Expression Language)
chain = prompt | llm | StrOutputParser()
#       ↑        ↑      ↑
#       formata  chama  extrai texto puro
```


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ─── PROVIDER SWITCHER LANGCHAIN ────────────────────────────────────────
# PROVIDER já está definido na célula de setup
# Troque lá e execute tudo novamente — o chain abaixo não muda

if PROVIDER == 'groq':
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model='llama-3.3-70b-versatile',
        temperature=0.3,
        api_key=userdata.get('GROQ_API_KEY')
    )

elif PROVIDER == 'openrouter':
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model='meta-llama/llama-3.3-70b-instruct',
        temperature=0.3,
        base_url='https://openrouter.ai/api/v1',
        api_key=userdata.get('OPENROUTER_API_KEY')
    )

print(f'✅ LangChain configurado: {PROVIDER}')


In [ ]:
# Chain simples: prompt -> llm -> texto puro
prompt_lc = ChatPromptTemplate.from_messages([
    ('system', 'Você é o SupportBot. Responda em português. Seja direto e empático.'),
    ('user',   '{mensagem_cliente}'),
])

chain = prompt_lc | llm | StrOutputParser()

# invoke() executa a chain inteira
resposta = chain.invoke({'mensagem_cliente': 'O app não abre desde ontem'})
print('Resposta:', resposta)
print('Tipo:', type(resposta))  # str puro — pronto para usar


---
### SupportBot completo com LangChain

Aqui reunimos **tudo que construímos na live**: lambda, try/except, list comprehension, zip e a chain LangChain.
O código funciona com Groq ou OpenRouter — basta trocar `PROVIDER` na célula de setup.


In [ ]:
normalizar_lc = lambda t: t.strip().lower()   # lambda

def suporte_langchain(mensagem_raw: str) -> str:   # função + try/except
    try:
        return chain.invoke({'mensagem_cliente': normalizar_lc(mensagem_raw)})
    except Exception as e:
        return f'[{PROVIDER}] Erro: {e}'


# Processar fila com list comprehension
fila_lc = [
    '  MEU APP NÃO ABRE  ',
    'Fui cobrado duas vezes este mês',
    'Quero cancelar minha conta',
]

respostas_lc = [suporte_langchain(m) for m in fila_lc]   # list comprehension

# Exibir com zip()
print(f'=== SUPPORTBOT LANGCHAIN [{PROVIDER.upper()}] ===')
for msg, resp in zip(fila_lc, respostas_lc):
    print(f'\n>>> {msg.strip()}')
    print(f'... {resp[:150]}')


---
## ✅ Resumo da Semana 03


## O que construímos hoje

| Conceito | Papel no agente | Onde apareceu |
|----------|-----------------|---------------|
| **Provider Switcher** | Portabilidade Groq ↔ OpenRouter | Setup |
| **Funções** | Pipeline classificar → responder | Seção 1 |
| **Lambda** | Normalização de input | Seção 2 |
| **Listas / Tuplas** | Histórico / Configuração imutável | Seção 3 |
| **List Comprehension** | Few-shot dinâmico, filtrar histórico | Seção 4 |
| **Dict Comprehension** | Roteamento intenção → system prompt | Seção 5 |
| **zip()** | Avaliação em batch | Seção 6 |
| **try/except/finally** | Resiliência com retry e fallback | Seção 7 |
| **System Prompt** | Personalidade permanente do agente | IA S03 |
| **LangChain LCEL** | `prompt | llm | parser` portável | LangChain |

---

## Exercícios para a semana

**1. Provider Switcher (iniciante)**
Crie conta no [openrouter.ai](https://openrouter.ai). Troque `PROVIDER='openrouter'` na célula de setup
e execute todo o notebook. As respostas mudaram? O tempo de resposta foi diferente?

**2. Nova intenção (intermediário)**
Adicione a intenção `'ONBOARDING'` ao banco de exemplos (seção 4) e ao dict de instruções (seção 5).
Teste com mensagens como `'Como começo a usar o produto?'`.
O classificador e o roteador devem funcionar sem mais nenhuma outra alteração.

**3. SupportBot LangChain com roteamento (avançado)**
Reimplemente o roteamento da seção 5 usando LangChain.
Dica: use `RunnableLambda` para chamar `classificar_com_fewshot()`,
depois `RunnableBranch` para selecionar o template por intenção.
Teste com `PROVIDER='groq'` e `PROVIDER='openrouter'`.

---

**Links úteis:**
- [Groq Console](https://console.groq.com)
- [OpenRouter](https://openrouter.ai) — modelos gratuitos em `/models?q=free`
- [LangChain LCEL](https://python.langchain.com/docs/expression_language/)
- [Tokenizer OpenAI](https://platform.openai.com/tokenizer)
